In [0]:
import pandas as pd

In [0]:
print("")

In [0]:
df = pd.read_csv("/Workspace/Shared/DS_Miniproject/data/merged/final_latest_calls_fullcols.csv")
df.info()

df_copy = df.copy()

In [0]:
df = df[df['Prospect_Outcome'] != 'Open']

In [0]:
print(df['Prospect_Outcome'].unique())
len(df.columns)

In [0]:
print(len(df))

In [0]:
null_percent = (df.isnull().sum() / len(df)) * 100
null_percent = null_percent.sort_values(ascending=False)

null_dict = null_percent.to_dict()
print(null_dict)

In [0]:
null_percent = (df.isnull().sum() / len(df)) * 100
null_percent = null_percent.sort_values(ascending=False)

# filter > 75%
high_null_dict = null_percent[null_percent > 70].to_dict()

print(high_null_dict)

In [0]:
cols_drop_95 = [col for col, val in high_null_dict.items() if val > 95]
df = df.drop(columns=cols_drop_95)

In [0]:
import numpy as np

# ── 1. Explicit Switching Intent ─────────────────────────────
col = 'Explicit_Switching_Intent_renewal'
df[col] = (
    df[col]
    .replace(np.nan, 'Unknown')   # handle NULL first
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        'yes': 'Yes',
        'no': 'No',
        'unknown': 'Unknown'
    })
    .fillna('Unknown')
)


# ── 2. Discount Offered ─────────────────────────────────────
col = 'Discount_Offered_renewal'
df[col] = (
    df[col]
    .replace(np.nan, 'Unknown')
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        'yes': 'Yes',
        'no': 'No',
        'unknown': 'Unknown'
    })
    .fillna('Unknown')
)


# ── 3. Customer Response ────────────────────────────────────
col = 'Customer_Response_renewal'
df[col] = (
    df[col]
    .replace(np.nan, 'Unknown')
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        'positive': 'Positive',
        'negative': 'Negative',
        'neutral': 'Neutral',
        'unknown': 'Unknown'
    })
    .fillna('Unknown')
)


# ── 4. Desire To Cancel ─────────────────────────────────────
col = 'Desire_To_Cancel_renewal'
df[col] = (
    df[col]
    .replace(np.nan, 'Unknown')
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        'renewed': 'Renew',
        'renew': 'Renew',
        'desired to cancel': 'Cancel',
        'unknown': 'Unknown'
    })
    .fillna('Unknown')
)



import numpy as np

# Generic cleaner + encoder
def binary_encode(col, mapping):
    return (
        df[col]
        .replace(np.nan, 'Unknown')
        .astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
        .fillna(0)   # Unknown → 0 (no signal)
    )


# ── 1. Explicit Switching Intent ─────────────────────────────
df['Explicit_Switching_Intent_flag'] = binary_encode(
    'Explicit_Switching_Intent_renewal',
    {
        'yes': 1,
        'no': 0,
        'unknown': 0
    }
)


# ── 2. Discount Offered ─────────────────────────────────────
df['Discount_Offered_flag'] = binary_encode(
    'Discount_Offered_renewal',
    {
        'yes': 1,
        'no': 0,
        'unknown': 0
    }
)


# ── 3. Customer Response (convert to NEGATIVE signal) ────────
df['Customer_Response_flag'] = binary_encode(
    'Customer_Response_renewal',
    {
        'negative': 1,   # churn signal
        'positive': 0,
        'neutral': 0,
        'unknown': 0
    }
)


# ── 4. Desire To Cancel ─────────────────────────────────────
df['cancel_flag'] = binary_encode(
    'Desire_To_Cancel_renewal',
    {
        'desired to cancel': 1,
        'cancel': 1,
        'renew': 0,
        'renewed': 0,
        'unknown': 0
    }
)

df = df.drop(columns=[
    'Explicit_Switching_Intent_renewal',
    'Discount_Offered_renewal',
    'Customer_Response_renewal',
    'Desire_To_Cancel_renewal'
])

In [0]:
print(df['cancel_flag'].value_counts())

In [0]:
important_sparse_cols = [
    col for col, val in high_null_dict.items()
    if 80 < val <= 95
]

print(important_sparse_cols)

In [0]:
# Drop known useless columns
drop_cols = [
    # 'Co_Ref',
    'Call_ID_renewal',
    'Contact_ID_cc',
    'Call_Date_renewal',
    'Prospect_Renewal_Date'
]

df = df.drop(columns=[col for col in drop_cols if col in df.columns])

In [0]:
len(df.columns)
print(df.columns)

In [0]:
drop_cols += [
    'Starting_Vat',
    'Starting_Gross',
    'Starting_Membership_Net',
    'Starting_Package_Net',
    'Starting_PQQ_Net',
    'Starting_Net',
    'Gross',
    'Membership_Net',
    'Package_Net',
    'PQQNet',
    'Amount'
]

df = df.drop(columns = drop_cols , errors='ignore')

In [0]:
# keep_cat = [
#     'Payment_Method',
#     'Band',
#     'Connection_Group',
#     'Tenure_Group',
#     'Anchor_Group'
# ]

len(df.columns)

print(df.columns)

In [0]:
df['target'] = df['Prospect_Outcome'].map({
    'Won': 0,
    'Churned': 1
})

df = df.drop(columns=['Prospect_Outcome'])

In [0]:
drop_cols = [
    'Call_Number_renewal'
]

In [0]:
# Convert to datetime
df['Closed_Date'] = pd.to_datetime(df['Closed_Date'], errors='coerce')
df['Registration_Date'] = pd.to_datetime(df['Registration_Date'], errors='coerce')

# Create feature
df['days_since_registration'] = (
    df['Closed_Date'] - df['Registration_Date']
).dt.days

In [0]:
date_cols = [
    'Proforma_Date',
    'Registration_Date',
    'Closed_Date',
    'Last_Renewal',
    'Prev_Renewal_Date',
    'DateTime_Out',
    # 'Payment_Method',
]
df = df.drop(columns=date_cols)

In [0]:
print(len(df.columns))

print(df.columns)


In [0]:
# Create encoded column
df['Call_attend_status_renewal'] = (
    df['Call_Direction_renewal']
    .fillna('unknown')
    .astype(str)
    .str.strip()
    .str.lower()
    .apply(lambda x: 0 if x == 'inbound' else 1)
)

# Crosstab using NEW column
print(df['Call_attend_status_renewal'].value_counts())

# Now drop old column (optional)
df = df.drop(columns=['Call_Direction_renewal'], errors='ignore')

In [0]:
import matplotlib.pyplot as plt

cat_cols = [
    'Payment_Method',
    'Band',
    'Connection_Group',
    'Tenure_Group',
    'Anchor_Group',
]

for col in cat_cols:
    print(f"\n==== {col} ====\n")
    
    # ── Handle missing values (important) ────────────────
    df[col] = df[col].fillna('Unknown')
    
    # ── Crosstab (% distribution) ───────────────────────
    ct = pd.crosstab(df[col], df['target'], normalize='index') * 100
    print(ct.round(2))
    
    # ── Add count (VERY IMPORTANT for interpretation) ───
    count = df[col].value_counts()
    print("\nCounts:\n", count)
    
    # ── Churn Rate ─────────────────────────────────────
    churn_rate = (
        df.groupby(col)['target']
        .mean()
        .sort_values(ascending=False)
    )
    
    # ── Plot ───────────────────────────────────────────
    plt.figure()
    churn_rate.plot(kind='bar')
    plt.title(f"Churn Rate by {col}")
    plt.ylabel("Churn Rate")
    plt.xlabel(col)
    plt.xticks(rotation=45)
    plt.show()

In [0]:
from scipy.stats import chi2_contingency

cat_cols = [
    'Payment_Method',
    'Band',
    'Connection_Group',
    'Tenure_Group',
    'Anchor_Group',
   
]

results = []

for col in cat_cols:
    # Create contingency table
    table = pd.crosstab(df[col], df['target'])
    
    # Apply Chi-square test
    chi2, p, dof, expected = chi2_contingency(table)
    
    results.append((col, chi2, p))

# Convert to DataFrame
res_df = pd.DataFrame(results, columns=['Feature', 'Chi2', 'p_value'])

# Sort by significance
res_df = res_df.sort_values('p_value')

print(res_df)

In [0]:
print(df.columns)

In [0]:
cols = [ '#_of_Connection', 'Last_Connections', 'Connection_Group' ]

In [0]:
import matplotlib.pyplot as plt

churn_rate = df.groupby('#_of_Connection')['target'].mean().sort_index()

plt.figure()
churn_rate.plot(kind='bar')
plt.title("Churn Rate by #_of_Connection")
plt.ylabel("Churn Rate")
plt.xlabel("#_of_Connection")
plt.xticks(rotation=45)
plt.show()

In [0]:
churn_rate = df.groupby('Last_Connections')['target'].mean().sort_index()

plt.figure()
churn_rate.plot(kind='bar')
plt.title("Churn Rate by Last_Connections")
plt.ylabel("Churn Rate")
plt.xlabel("Last_Connections")
plt.xticks(rotation=45)
plt.show()

In [0]:
churn_rate = df.groupby('Connection_Group')['target'].mean().sort_values(ascending=False)

plt.figure()
churn_rate.plot(kind='bar')
plt.title("Churn Rate by Connection_Group")
plt.ylabel("Churn Rate")
plt.xlabel("Connection_Group")
plt.xticks(rotation=45)
plt.show()

In [0]:
from scipy.stats import pointbiserialr

cols = ['#_of_Connection', 'Last_Connections']

for col in cols:
    print(f"\n==== {col} ====\n")
    
    # Drop nulls just in case
    temp = df[[col, 'target']].dropna()
    
    corr, p = pointbiserialr(temp['target'], temp[col])
    
    print("Correlation:", round(corr, 4))
    print("p-value:", p)

In [0]:
df = df.drop(columns=['#_of_Connection', 'Last_Connections' , 'Payment_Method'])

In [0]:
len(df.columns)

print(df.columns)

In [0]:
print(df['Membership_Renewal_Decision_renewal'].value_counts())

In [0]:
import matplotlib.pyplot as plt

# Clean column
df['Membership_Renewal_Decision_renewal'] = (
    df['Membership_Renewal_Decision_renewal']
    .fillna('Unknown')
    .astype(str)
    .str.strip()
)

# Churn rate
churn_rate = (
    df.groupby('Membership_Renewal_Decision_renewal')['target']
    .mean()
    .sort_values(ascending=False)
)

plt.figure()
churn_rate.plot(kind='bar')
plt.title("Churn Rate by Membership_Renewal_Decision")
plt.ylabel("Churn Rate")
plt.xlabel("Membership_Renewal_Decision")
plt.xticks(rotation=45)
plt.show()

In [0]:
from scipy.stats import chi2_contingency

table = pd.crosstab(
    df['Membership_Renewal_Decision_renewal'],
    df['target']
)

chi2, p, dof, expected = chi2_contingency(table)

print("Chi2:", chi2)
print("p-value:", p)

In [0]:
# Yes/No → ~100% churn
# Yes → ~70% churn
# Unknown → ~10%
# No → ~5%

# This is an extremely strong separation

In [0]:
df = df.drop(columns=['Membership_Renewal_Decision_renewal'], errors='ignore')

In [0]:
df['Analysed_Call_renewal'].value_counts()


In [0]:
df = df.drop(columns=['Analysed_Call_renewal'])

In [0]:

print(df['Last_Band'].value_counts())
print(df['Band'].value_counts())

In [0]:
cols = [
    'Mentioned_Competitors_renewal',
    'Competitor_Benefits_Mentioned_renewal',
    'Topic_Introduced_By_renewal'
]

for col in cols:
    print(f"\n==== {col} ====")
    print(df[col].value_counts(dropna=False))

In [0]:
# # ───────────────────────────────────────────────────────────
# # Handle competitor + topic columns (final version)
# # ───────────────────────────────────────────────────────────

# # 1. Mentioned Competitors → flag
# df['competitor_mentioned_flag'] = df['Mentioned_Competitors_renewal'].notnull().astype(int)

# # 2. Drop messy high-cardinality column
# df = df.drop(columns=['Competitor_Benefits_Mentioned_renewal'], errors='ignore')

# # 3. Topic Introduced By → clean + encode
# df['Topic_Introduced_By_renewal'] = df['Topic_Introduced_By_renewal'].fillna('Unknown')

# df = pd.get_dummies(
#     df,
#     columns=['Topic_Introduced_By_renewal'],
#     drop_first=True
# )

# # 4. Drop original competitor column
# df = df.drop(columns=['Mentioned_Competitors_renewal'], errors='ignore')

# # ───────────────────────────────────────────────────────────
# # Done: 
# # - competitor_mentioned_flag created
# # - noisy column removed
# # - topic column encoded
# # ───────────────────────────────────────────────────────────

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import chi2_contingency

cols = [
    'Mentioned_Competitors_renewal',
    'Competitor_Benefits_Mentioned_renewal',
    'Topic_Introduced_By_renewal'
]

results = []

for col in cols:
    
    if col not in df.columns:
        print(f"\nSkipping {col} (not in df)")
        continue
    
    print(f"\n==== {col} ====")
    
    # Crosstab
    ct = pd.crosstab(df[col].fillna('NaN'), df['target'], normalize='index') * 100
    print(ct.round(2))
    
    # Plot top 10
    top_vals = df[col].fillna('NaN').value_counts().head(10).index
    ct_top = ct.loc[top_vals]
    
    ct_top.plot(kind='bar')
    plt.title(f'{col} vs Target')
    plt.xlabel(col)
    plt.ylabel('Percentage')
    plt.xticks(rotation=45)
    plt.show()
    
    # Chi-square
    contingency = pd.crosstab(df[col].fillna('NaN'), df['target'])
    chi2, p, dof, expected = chi2_contingency(contingency)
    
    results.append({
        'Feature': col,
        'Chi2_Score': chi2,
        'p_value': p
    })

chi_df = pd.DataFrame(results).sort_values(by='Chi2_Score', ascending=False)

print("\n=== Chi-Square Results ===")
print(chi_df)

In [0]:
# competitor signal
df['competitor_mentioned_flag'] = df['Mentioned_Competitors_renewal'].notnull().astype(int)

# topic signal
df['topic_customer_flag'] = (df['Topic_Introduced_By_renewal'] == 'Customer').astype(int)

# optional
df['topic_agent_flag'] = (df['Topic_Introduced_By_renewal'] == 'Agent').astype(int)

In [0]:
df = df.drop(columns=[
    'Competitor_Benefits_Mentioned_renewal',
    'Mentioned_Competitors_renewal',
    'Topic_Introduced_By_renewal'
], errors='ignore')

In [0]:
# When a customer mentions competitors, it usually means:

# Comparing alternatives
# Evaluating switching
# Price dissatisfaction
# Value concerns

# All are strong churn signals

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

cols = [
    'Prospect_Status',
    'Proforma_Account_Stage',
    'Proforma_Audit_Status'
]

for col in cols:
    
    if col not in df.columns:
        print(f"\n Skipping {col}")
        continue
    
    print(f"\n==== {col} ====")
    
    # Take top 10 categories for readability
    top_vals = df[col].fillna('NaN').value_counts().head(10).index
    
    # Crosstab (%)
    ct = pd.crosstab(
        df[col].fillna('NaN'),
        df['target'],
        normalize='index'
    ) * 100
    
    ct_top = ct.loc[top_vals]
    
    print(ct_top.round(2))
    
    # Plot
    ct_top.plot(kind='bar')
    plt.title(f'{col} vs Target (Top Categories)')
    plt.xlabel(col)
    plt.ylabel('Percentage')
    plt.xticks(rotation=45)
    plt.show()

In [0]:
cols = [
    # 'Membership_Renewal_Decision_renewal',
    'Prospect_Status',
    'Proforma_Account_Stage',
    'Proforma_Audit_Status'
]

for col in cols:
     print(df[col].value_counts())
     print()

In [0]:
df['Proforma_Account_Stage'] = df['Proforma_Account_Stage'].str.strip().str.title()

counts = df['Proforma_Account_Stage'].value_counts()

rare = counts[counts < 100].index

df['Proforma_Account_Stage'] = df['Proforma_Account_Stage'].replace(rare, 'Other')

In [0]:
df['stage_high_risk_flag'] = df['Proforma_Account_Stage'].isin([
    'Membership Only',
    'Vetting'
]).astype(int)

df['stage_low_risk_flag'] = (df['Proforma_Account_Stage'] == 'Published').astype(int)

In [0]:
df = df.drop(columns=['Prospect_Status'])
df = df.drop(columns=['Proforma_Account_Stage'])

In [0]:
print(len(df.columns))

In [0]:
print(df.columns)
print(len(df.columns))

In [0]:
extra_in_df = set(df.columns) - set(df_copy.columns)
print("Extra in df:", extra_in_df)

In [0]:
dropped_cols = list(set(df_copy.columns) - set(df.columns))
print("Dropped columns:", dropped_cols)

In [0]:
# redundancy in total agent flag 


df = df.drop(columns=['topic_agent_flag'], errors='ignore')

# These two are correlated / redundant

# Because:

# If customer = 1 → agent = 0
# If agent = 1 → customer = 0

# Model gets duplicate information


In [0]:
extra_in_df = set(df.columns) - set(df_copy.columns)
# print("Extra in df:", extra_in_df)

print(set(df.columns) - set(extra_in_df))
print(len(set(df.columns) - set(extra_in_df)))

In [0]:
print(df['Call_Number_renewal'].value_counts())

In [0]:
x = [
    'Call_Number_renewal',
    'Call_Year_renewal',
]

df = df.drop(columns = x)

In [0]:
x = [
    'Renewal_Month',
    'Renewal_Year'
]

df = df.drop(columns = x)



In [0]:
extra_in_df = set(df.columns) - set(df_copy.columns)
# print("Extra in df:", extra_in_df)

print(set(df.columns) - set(extra_in_df))
print(len(set(df.columns) - set(extra_in_df)))

In [0]:
print(len(df.columns))

In [0]:
print(df['Payment_Timeframe'].value_counts())

In [0]:
df['Payment_Timeframe'] = np.sign(df['Payment_Timeframe']) * np.log1p(abs(df['Payment_Timeframe']))

In [0]:
print(df['Payment_Timeframe'].value_counts())

# =============================================================================================

In [0]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()
print(num_cols)
print(len(num_cols))

In [0]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(cat_cols)
print(len(cat_cols))

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = df.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr,
    annot=True,              # keep only one
    cmap='coolwarm',
    center=0,
    linewidths=0.5
)

plt.title("Correlation Heatmap")
plt.show()

In [0]:
corr_target = df.corr(numeric_only=True)[['target']].sort_values(by='target', ascending=False)

plt.figure(figsize=(6, 10))
sns.heatmap(
    corr_target,
    annot=True,
    cmap='coolwarm',
    center=0
)

plt.title("Correlation with Target")
plt.show()

In [0]:
import numpy as np

corr_matrix = df.corr(numeric_only=True).abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Highly correlated:", high_corr)

In [0]:
df['pct_inc_total_net_paid'] = (
    (df['Total_Net_Paid'] - df['Last_Total_Net_Paid']) 
    / (df['Last_Total_Net_Paid'] + 1e-5)
)

In [0]:
# 1. Financial cluster (VERY CLEAR)
drop = ['Total_Amount', 'Last_Total_Net_Paid']

df = df.drop(columns=drop, errors='ignore')



# These are almost duplicates

# ✔ Keep ONLY 1–2

In [0]:
# Highly correlated with:

# Tenure_Years
# ✔ Keep ONLY ONE

# Prefer:

keep = ['Tenure_Years']
drop = ['days_since_registration']

df.drop(columns=drop, errors='ignore', inplace=True)

# Easier interpretation + cleaner

In [0]:
print(len(df.columns))

In [0]:
from scipy.stats import chi2_contingency



results = []

for col in cat_cols:
    table = pd.crosstab(df[col], df['target'])
    
    chi2, p, _, _ = chi2_contingency(table)
    
    results.append((col, chi2, p))

chi_df = pd.DataFrame(results, columns=['Feature', 'Chi2', 'p_value'])
chi_df = chi_df.sort_values('p_value')

print(chi_df)

In [0]:
# [
#     'Discount_Amount',
#     'Current_World_Pay_Token',
#     'Current_Auto_Renewal_Flag',
#     'Proforma_Audit_Status',              # possible leakage
#     'Proforma_Membership_Status',         # likely leakage
#     'Connection_Group',
#     'Anchor_Group',
#     'Tenure_Group',
#     'Band',
#     'Agent_Response_Category_renewal',
#     'Customer_Renewal_Response_Category_renewal'
# ]




# 2. STRONG BUT CHECK FOR LEAKAGE
# [
#     'Proforma_Audit_Status',
#     'Proforma_Membership_Status'
# ]

# These are too strong → suspicious

# Recommendation:
# drop_cols += [
#     'Proforma_Audit_Status',
#     'Proforma_Membership_Status'
# ]

In [0]:
# From your Chi-square:

# Very low Chi2
# High p-values
# Weak signal

# That means:

# They are weak predictors, regardless of direction


df = df.drop(columns=[
    'Proforma_Audit_Status',
    'Proforma_Membership_Status',
    'Discussion_on_Price_Increase_renewal',
    'Agent_Flagged_Membership_Status_Alert_renewal',
    'Monetary_Price_Increase_Mentioned_renewal',
    'Customer_Asked_For_Justification_renewal',
    'Percentage_Price_Increase_Mentioned_renewal',
    'Price_Range_Mentioned_renewal'
], errors='ignore')

In [0]:
print(len(df.columns))

In [0]:
print(len(df.columns))
print(df.columns)

In [0]:
df = df.drop(columns=['Last_Band'])

In [0]:
print(len(df.columns))

print(df.columns)

In [0]:
import matplotlib.pyplot as plt

churn = df.groupby('Anchor_Group')['target'].mean().sort_values()

plt.figure()
churn.plot(kind='bar')
plt.title("Churn Rate by Anchor_Group")
plt.ylabel("Churn Rate")
plt.xlabel("Anchor_Group")
plt.xticks(rotation=45)
plt.show()

In [0]:
# df = df.drop(columns=['Current_Anchorings'])

In [0]:
# days since reg vs tenure score  both are same in general

df = df.drop(columns=['Tenure_Scores'])


In [0]:
if 'days_since_registration' in df.columns:
    print("Column exists")
else:
    print("Column not found")

In [0]:
print(len(df.columns))

In [0]:
extra_in_df = set(df.columns) - set(df_copy.columns)
# print("Extra in df:", extra_in_df)

print(set(df.columns) - set(extra_in_df))
print(len(set(df.columns) - set(extra_in_df)))

In [0]:
ct = pd.crosstab(
    df['Proforma_World_Pay_Token'],
    df['target'],
    normalize='index'
) * 100

print(ct.round(2))

In [0]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df['Proforma_World_Pay_Token'], df['target'])

chi2, p, _, _ = chi2_contingency(table)

print("Chi2:", chi2)
print("p-value:", p)

In [0]:
diff = df.groupby('Proforma_World_Pay_Token')['target'].mean().diff().iloc[-1]
print("Difference:", diff)

In [0]:
# Metric	Result	Decision
# p-value	Significant misleading
# Difference	~0.7% weak
# Signal strength	Very low useless

In [0]:
df = df.drop(columns=['Proforma_World_Pay_Token'])

In [0]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df['Proforma_Auto_Renewal'], df['target'])

chi2, p, _, _ = chi2_contingency(table)

print("Chi2:", chi2)
print("p-value:", p)

In [0]:
diff = df.groupby('Proforma_Auto_Renewal')['target'].mean().diff().iloc[-1]
print("Difference:", diff)


In [0]:
print(len(df.columns))
print(df.columns)

In [0]:
df = df.drop(columns=['Total_Renewal_Score_New'])

In [0]:
print(df['Proforma_Approved_Lists'].value_counts())
print(df['Current_Anchorings'].value_counts())

df = df.drop(columns=['Proforma_Approved_Lists'])



In [0]:
len(df.columns)

In [0]:
print(df.columns)

In [0]:
df['Agent_Renewal_Pitch_Category_renewal'].isnull().sum()

df.drop(columns=['Agent_Renewal_Pitch_Category_renewal'], inplace=True)

In [0]:
len(df)

In [0]:
df['Call_Reschedule_Request_renewal'].value_counts()

ct = pd.crosstab(
    df['Call_Reschedule_Request_renewal'],
    df['target'],
    normalize='index'
) * 100

print(ct.round(2))

In [0]:
df['Agent_Renewal_Initiation_renewal'].value_counts()
ct = pd.crosstab(
    df['Agent_Renewal_Initiation_renewal'],
    df['target'],
    normalize='index'
) * 100

print(ct.round(2))

In [0]:
# Metric	Value	Meaning
# Difference	~1.1%	Very weak


df = df.drop(columns=['Agent_Renewal_Initiation_renewal'])

In [0]:
# Still Reduce
# https://chatgpt.com/s/t_69dd0783f9f48191b440da3615b3d47a

In [0]:
print(df['Other_Complaint_renewal'].value_counts())

ct = pd.crosstab(
    df['Other_Complaint_renewal'],
    df['target'],
    normalize='index'
) * 100

print(ct.round(2))

In [0]:
# Group	Churn
# No complaint	8.19%
# Yes complaint	15.45%

# Difference = 7.26%

# No group:
# 13617 × 8.19% ≈ 1115 churners
# Yes group:
# 2681 × 15.45% ≈ 414 churners


In [0]:
print(df.columns)
print(len(df.columns))

# Cleaning and Check 

In [0]:
for col in df.columns:
    print(col, df[col].isnull().sum())

In [0]:
df['Customer_Renewal_Response_Category_renewal'].value_counts()

In [0]:
pd.crosstab(
    df['Customer_Renewal_Response_Category_renewal'],
    df['target'],
    normalize='index'
) * 100

In [0]:
def map_risk_improved(x):
    if x in ['Cancellation / Termination / Non-Renewal / Withdrawal',
             'Financial Hardship / Struggles']:
        return 'High Risk'
    
    elif x in ['Confusion / Uncertainty / Hesitation',
               'Price and Cost',
               'Quality / Value / Satisfaction of service']:
        return 'Medium Risk'
    
    elif x in ['Agreement', 'Auto / Automatic',
               'Payments and Billings / Invoice']:
        return 'Low Risk'
    
    elif x == 'Missing':
        return 'Missing'
    
    else:
        return 'Neutral'   # instead of dumping into "Other"

df['Customer_Response_Risk'] = df['Customer_Renewal_Response_Category_renewal'].apply(map_risk_improved)

In [0]:
df  = df.drop(columns=['Customer_Renewal_Response_Category_renewal'])


In [0]:
pd.crosstab(df['Customer_Response_Risk'], df['target'])

In [0]:
print(df['Customer_Response_Risk'].isnull().sum())

In [0]:
def analyze_column(df, col):
    print(f"\n===== {col} =====")
    
    # Null stats
    nulls = df[col].isnull().sum()
    total = len(df)
    print(f"Null %: {round((nulls/total)*100,2)}%")
    
    # Value counts
    print("\nValue Counts:")
    print(df[col].value_counts(dropna=False).head(10))
    
    # Crosstab
    print("\nCrosstab (%):")
    print(pd.crosstab(df[col], df['target'], normalize='index') * 100)
high_null_cols = [
    # 'Agent_Response_Category_renewal',
    'Serious_Complaint_renewal',
    # 'Other_Complaint_renewal',
    'Renewal_Impact_Due_to_Price_Increase_renewal',
    # 'Discount_or_Waiver_Requested_renewal',
    'Call_Reschedule_Request_renewal',
    # 'Explicit_Competitor_Mention_renewal',
    # 'Price_Switching_Mentioned_renewal',
    'Competitor_Value_Comparison_renewal'
]

for col in high_null_cols:
    analyze_column(df , col)

In [0]:
df = df.drop(columns=['Other_Complaint_renewal'])

In [0]:
df['agent_complaint_flag'] = df['Agent_Response_Category_renewal'].isin([
    'Complaints',
    'Customer Service / Support'
]).astype(int)

In [0]:
df = df.drop(columns = ['Agent_Response_Category_renewal'])

In [0]:
df['Explicit_Competitor_Mention_renewal'] = df['Explicit_Competitor_Mention_renewal'].fillna('No')
df['Explicit_Competitor_Mention_renewal'] = \
    df['Explicit_Competitor_Mention_renewal'].apply(lambda x: 1 if x == 'Yes' else 0)

In [0]:
df['Price_Switching_Mentioned_renewal'] = df['Price_Switching_Mentioned_renewal'].fillna('No')
df['Price_Switching_Mentioned_renewal'] = \
    df['Price_Switching_Mentioned_renewal'].apply(lambda x: 1 if 'Yes' in str(x) else 0)

In [0]:
# Serious Complaint
df['Serious_Complaint_renewal'] = df['Serious_Complaint_renewal'].fillna('No').map({'No': 0, 'Yes': 1})

# Price Impact
df['Renewal_Impact_Due_to_Price_Increase_renewal'] = \
    df['Renewal_Impact_Due_to_Price_Increase_renewal'].fillna('No') \
    .apply(lambda x: 1 if 'Yes' in str(x) else 0)

# Call Reschedule
df['Call_Reschedule_Request_renewal'] = \
    df['Call_Reschedule_Request_renewal'].fillna('No').map({'No': 0, 'Yes': 1})

# Competitor Value
df['Competitor_Value_Comparison_renewal'] = \
    df['Competitor_Value_Comparison_renewal'].fillna('unknown')

In [0]:
# too sparse
# This feature is:

# Too sparse
# Weak separation
# Noisy text variations

# Raw column = NOT useful
df.drop(columns=['Discount_or_Waiver_Requested_renewal'], inplace=True)
# df.drop(columns=['Last_Years_Price '], inplace=True)


In [0]:
for col in df.columns:
    print(col , df[col].isnull().sum())

In [0]:
df['Current_Anchor_List'] = df['Current_Anchor_List'].fillna('None')

In [0]:
df['Proforma_Auto_Renewal'].value_counts()

In [0]:
df['Proforma_Auto_Renewal'] = (
    df['Proforma_Auto_Renewal']
    .astype(str)  # ensure string
    .apply(lambda x: 1 if 'yes' in x.lower() else 0)
)

In [0]:
df['Proforma_Auto_Renewal'].isnull().sum()

In [0]:
df = df.drop(columns='Last_Years_Price')

In [0]:
# no need pct_increse while already have total_net_paid
df = df.drop(columns=['pct_inc_total_net_paid'])

In [0]:
df.drop(columns=['pct_inc_total_net_paid'], inplace=True, errors='ignore')

In [0]:
df['Payment_Timeframe'] = df['Payment_Timeframe'].fillna(
    df.groupby('Co_Ref')['Payment_Timeframe'].transform('median')
)
df['Payment_Timeframe'] = df['Payment_Timeframe'].fillna(
    df['Payment_Timeframe'].median()
)

In [0]:
# 2. Group-level (Co_Ref) median
group_median = df.groupby('Co_Ref')['Total_Net_Paid'].transform('median')

df['Total_Net_Paid'] = df['Total_Net_Paid'].fillna(group_median)

# 3. Fallback → global median
df['Total_Net_Paid'] = df['Total_Net_Paid'].fillna(df['Total_Net_Paid'].median())



In [0]:
df['Renewal_Score_At_Release'] = df['Renewal_Score_At_Release'].fillna(
    df.groupby('Co_Ref')['Renewal_Score_At_Release'].transform('median')
)

# fallback → global median
df['Renewal_Score_At_Release'] = df['Renewal_Score_At_Release'].fillna(
    df['Renewal_Score_At_Release'].median()
)

In [0]:
# Co_Ref median
df['Tenure_Years'] = df['Tenure_Years'].fillna(
    df.groupby('Co_Ref')['Tenure_Years'].transform('median')
)

# fallback → global median
df['Tenure_Years'] = df['Tenure_Years'].fillna(
    df['Tenure_Years'].median()
)

In [0]:
for col in df.columns:
    print(col , df[col].isnull().sum())

In [0]:
df.drop(columns=['Co_Ref'], inplace=True)

In [0]:
print(len(df.columns))

In [0]:
for col in df.columns:
    print(col , "---" , df[col].dtype , df[col].value_counts)

# Ready dataset finetune

In [0]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder


df['Discount_Amount'] = (
    df['Discount_Amount']
    .astype(str)
    .str.replace('%', '')
)

df['Discount_Amount'] = pd.to_numeric(df['Discount_Amount'], errors='coerce')


binary_map = {'y': 1, 'n': 0, 'yes': 1, 'no': 0}

binary_cols = [
    'Current_Auto_Renewal_Flag',
    'Current_World_Pay_Token'
]

for col in binary_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.lower()
        .map(binary_map)
        .fillna(0)
        .astype(int)
    )


df.drop(columns=['Current_Anchor_List'], inplace=True, errors='ignore')


# Convert remaining object → category
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')



num_cols = df.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()
cat_cols = df.select_dtypes(include=['category']).columns.tolist()

# Remove target from encoding
if 'target' in cat_cols:
    cat_cols.remove('target')

print("Numeric:", len(num_cols))
print("Categorical:", cat_cols)



for col in cat_cols:
    
    unique_vals = df[col].nunique()
    
    if unique_vals <= 20:
        # One-hot encode
        df = pd.get_dummies(df, columns=[col], drop_first=True)
        
    else:
        # Label encode
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))



print("\nRemaining categorical columns:")
print(df.select_dtypes(include='object').columns)

print("\nDataset shape:", df.shape)

In [0]:
corr = df.corr().abs()
high_corr = (corr > 0.9).sum().sort_values(ascending=False)
high_corr.head(10)

In [0]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum()
)
# model = XGBClassifier(
#     n_estimators=500,
#     max_depth=4,
#     learning_rate=0.03
# )

model.fit(X_train, y_train)
print(X_test , X_train)

In [0]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Compute F1 for each threshold
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

# Get best threshold
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best Threshold (F1):", best_threshold)
print("Best F1 Score:", f1_scores[best_idx])

In [0]:
# from sklearn.metrics import classification_report, roc_auc_score

# y_pred = model.predict(X_test)
# y_prob = model.predict_proba(X_test)[:, 1]

# print(classification_report(y_test, y_pred))
# print("ROC AUC:", roc_auc_score(y_test, y_prob))

from sklearn.metrics import classification_report, roc_auc_score

# Probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Custom threshold (change this to improve precision)
threshold = 0.05

# Predictions using custom threshold
y_pred = (y_prob > threshold).astype(int)

# Evaluation
print(f"Threshold used: {threshold}")
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

In [0]:
sample_non_churn_full = {
 'Discount_Amount': 0.0,
 'Sustainability_Score': 9.5,
 'Auto_Renewal_Score': 9,
 'Status_Scores': 9,
 'Anchoring_Score': 9.0,
 'Proforma_Auto_Renewal': 0,
 'Current_Anchorings': 2,
 'Payment_Timeframe': 0.0,
 'Current_Auto_Renewal_Flag': 1,
 'Current_World_Pay_Token': 1,
 'Renewal_Score_At_Release': 27.0,
 'Tenure_Years': 5,
 'Total_Net_Paid': 700,
 'Serious_Complaint_renewal': 0,
 'Renewal_Impact_Due_to_Price_Increase_renewal': 0,
 'Call_Reschedule_Request_renewal': 0,
 'Explicit_Competitor_Mention_renewal': 0,
 'Price_Switching_Mentioned_renewal': 0,
 'Explicit_Switching_Intent_flag': 0,
 'Discount_Offered_flag': 0,
 'Customer_Response_flag': 0,
 'cancel_flag': 0,
 'Call_attend_status_renewal': 1,
 'competitor_mentioned_flag': 0,
 'topic_customer_flag': 0,
 'stage_high_risk_flag': 0,
 'stage_low_risk_flag': 1,
 'agent_complaint_flag': 0,

 # Band
 'Band_Band B': 0, 'Band_Band C1': 0, 'Band_Band C2': 0, 'Band_Band D': 1,
 'Band_Band E': 0, 'Band_Band F': 0, 'Band_Band F1': 0, 'Band_Band F2': 0,
 'Band_Band G': 0, 'Band_Band H': 0, 'Band_Band I': 0, 'Band_Band J': 0,
 'Band_Group': 0, 'Band_Unknown': 0,

 # Connection
 'Connection_Group_10+': 0, 'Connection_Group_2': 0, 'Connection_Group_3': 1,
 'Connection_Group_4-9': 0, 'Connection_Group_Unknown': 0,
 'Connection_Group_independent': 0,

 # Tenure
 'Tenure_Group_2': 0, 'Tenure_Group_3': 0,
 'Tenure_Group_4+': 1, 'Tenure_Group_Unknown': 0,

 # Anchor
 'Anchor_Group_10+': 0, 'Anchor_Group_2': 0, 'Anchor_Group_3': 1,
 'Anchor_Group_4-9': 0, 'Anchor_Group_Unknown': 0,
 'Anchor_Group_independent': 0,

 # Competitor
 'Competitor_Value_Comparison_renewal_not discussed': 0,
 'Competitor_Value_Comparison_renewal_similar value': 0,
 'Competitor_Value_Comparison_renewal_unknown': 1,

 # Customer risk
 'Customer_Response_Risk_Low Risk': 1,
 'Customer_Response_Risk_Medium Risk': 0,
 'Customer_Response_Risk_Neutral': 0
}

sample_churn_full = {
 'Discount_Amount': 20.0,
 'Sustainability_Score': 8.0,
 'Auto_Renewal_Score': 8,
 'Status_Scores': 0,
 'Anchoring_Score': 7.5,
 'Proforma_Auto_Renewal': 0,
 'Current_Anchorings': 0,
 'Payment_Timeframe': 3.5,
 'Current_Auto_Renewal_Flag': 1,
 'Current_World_Pay_Token': 0,
 'Renewal_Score_At_Release': 24.0,
 'Tenure_Years': 1,
 'Total_Net_Paid': 400,
 'Serious_Complaint_renewal': 1,
 'Renewal_Impact_Due_to_Price_Increase_renewal': 1,
 'Call_Reschedule_Request_renewal': 1,
 'Explicit_Competitor_Mention_renewal': 1,
 'Price_Switching_Mentioned_renewal': 1,
 'Explicit_Switching_Intent_flag': 1,
 'Discount_Offered_flag': 1,
 'Customer_Response_flag': 1,
 'cancel_flag': 0,
 'Call_attend_status_renewal': 1,
 'competitor_mentioned_flag': 1,
 'topic_customer_flag': 1,
 'stage_high_risk_flag': 1,
 'stage_low_risk_flag': 0,
 'agent_complaint_flag': 1,

 # Band
 'Band_Band B': 0, 'Band_Band C1': 1, 'Band_Band C2': 0, 'Band_Band D': 0,
 'Band_Band E': 0, 'Band_Band F': 0, 'Band_Band F1': 0, 'Band_Band F2': 0,
 'Band_Band G': 0, 'Band_Band H': 0, 'Band_Band I': 0, 'Band_Band J': 0,
 'Band_Group': 0, 'Band_Unknown': 0,

 # Connection
 'Connection_Group_10+': 0, 'Connection_Group_2': 1, 'Connection_Group_3': 0,
 'Connection_Group_4-9': 0, 'Connection_Group_Unknown': 0,
 'Connection_Group_independent': 0,

 # Tenure
 'Tenure_Group_2': 1, 'Tenure_Group_3': 0,
 'Tenure_Group_4+': 0, 'Tenure_Group_Unknown': 0,

 # Anchor
 'Anchor_Group_10+': 0, 'Anchor_Group_2': 1, 'Anchor_Group_3': 0,
 'Anchor_Group_4-9': 0, 'Anchor_Group_Unknown': 0,
 'Anchor_Group_independent': 0,

 # Competitor
 'Competitor_Value_Comparison_renewal_not discussed': 0,
 'Competitor_Value_Comparison_renewal_similar value': 1,
 'Competitor_Value_Comparison_renewal_unknown': 0,

 # Customer risk
 'Customer_Response_Risk_Low Risk': 0,
 'Customer_Response_Risk_Medium Risk': 1,
 'Customer_Response_Risk_Neutral': 0
}
sample_churn_fixed = sample_churn_full.copy()

# Make it clearly churn-like
sample_churn_fixed.update({
    'Serious_Complaint_renewal': 1,
    'Renewal_Impact_Due_to_Price_Increase_renewal': 1,
    'Price_Switching_Mentioned_renewal': 1,
    'Explicit_Switching_Intent_flag': 1,
    'stage_high_risk_flag': 1,
    
    # IMPORTANT
    'Customer_Response_Risk_Low Risk': 0,
    'Customer_Response_Risk_Medium Risk': 0,
    'Customer_Response_Risk_Neutral': 0,
    # (implicitly High Risk pattern)
})


sample_churn_strong = {
 'Discount_Amount': 50.0,
 'Sustainability_Score': 6.0,   # ↓ low
 'Auto_Renewal_Score': 5,       # ↓ low
 'Status_Scores': 0,
 'Anchoring_Score': 6.5,

 'Proforma_Auto_Renewal': 0,
 'Current_Anchorings': 0,
 'Payment_Timeframe': 5.0,

 # ❗ REMOVE retention
 'Current_Auto_Renewal_Flag': 0,
 'Current_World_Pay_Token': 0,

 'Renewal_Score_At_Release': 20.0,
 'Tenure_Years': 1,
 'Total_Net_Paid': 200,

 # strong churn signals
 'Serious_Complaint_renewal': 1,
 'Renewal_Impact_Due_to_Price_Increase_renewal': 1,
 'Call_Reschedule_Request_renewal': 1,
 'Explicit_Competitor_Mention_renewal': 1,
 'Price_Switching_Mentioned_renewal': 1,
 'Explicit_Switching_Intent_flag': 1,

 'Discount_Offered_flag': 1,
 'Customer_Response_flag': 1,

 'cancel_flag': 0,
 'Call_attend_status_renewal': 0,

 'competitor_mentioned_flag': 1,
 'topic_customer_flag': 1,

 'stage_high_risk_flag': 1,
 'stage_low_risk_flag': 0,

 'agent_complaint_flag': 1,

 # categories
 'Band_Band B': 0, 'Band_Band C1': 1, 'Band_Band C2': 0, 'Band_Band D': 0,
 'Band_Band E': 0, 'Band_Band F': 0, 'Band_Band F1': 0, 'Band_Band F2': 0,
 'Band_Band G': 0, 'Band_Band H': 0, 'Band_Band I': 0, 'Band_Band J': 0,
 'Band_Group': 0, 'Band_Unknown': 0,

 'Connection_Group_10+': 0, 'Connection_Group_2': 1, 'Connection_Group_3': 0,
 'Connection_Group_4-9': 0, 'Connection_Group_Unknown': 0,
 'Connection_Group_independent': 0,

 'Tenure_Group_2': 1, 'Tenure_Group_3': 0,
 'Tenure_Group_4+': 0, 'Tenure_Group_Unknown': 0,

 'Anchor_Group_10+': 0, 'Anchor_Group_2': 1, 'Anchor_Group_3': 0,
 'Anchor_Group_4-9': 0, 'Anchor_Group_Unknown': 0,
 'Anchor_Group_independent': 0,

 'Competitor_Value_Comparison_renewal_not discussed': 0,
 'Competitor_Value_Comparison_renewal_similar value': 1,
 'Competitor_Value_Comparison_renewal_unknown': 0,

 'Customer_Response_Risk_Low Risk': 0,
 'Customer_Response_Risk_Medium Risk': 1,
 'Customer_Response_Risk_Neutral': 0
}

test_df = pd.DataFrame([sample_non_churn_full, sample_churn_fixed , sample_churn_strong])

preds = model.predict(test_df)
probs = model.predict_proba(test_df)[:, 1]

test_df['prediction'] = preds
test_df['probability'] = probs

print(test_df[['prediction', 'probability']])

In [0]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
top_features = importance.sort_values(ascending=False).head(20)

print(top_features)

# ====================================================================

In [0]:
# import shap

# explainer = shap.Explainer(model)
# shap_values = explainer(X_test)

# shap.plots.bar(shap_values)

# tune

In [0]:
# model = XGBClassifier(
#     n_estimators=500,
#     max_depth=4,
#     learning_rate=0.03
# )

In [0]:
# low_importance = importance[importance < 0.001].index
# df.drop(columns=low_importance, inplace=True)